# 49 · Where human and mouse PT agree: conservation of gene gradients by gradient strength

Referee report on draft v0.3, items 2(a), 2(b), F3 and M2. The title claims that human and mouse
proximal tubule share the zonation of their most strongly zonated genes and little else. Notebook
43's conservation index (cross-species agreement of gene gradients relative to same-species,
cross-dataset ceilings) is low overall. Is that because agreement sits only in the strongest
gradients? And in the weaker genes, is the low agreement a species difference or an absence of
reproducible zonation? Part B tests how sensitive the index is to its construction.

Everything uses segment labels (the reviewed labels) and segment pseudobulks, as notebook 43. Two
male control mice and two cortex sections from one male human donor; descriptive.

## Protocol (fixed in `SCRATCH/ws6/plan.md` before any result of this notebook was computed)

**Guard.** Notebook 43's inputs are rebuilt with its code and constants, and its published
`conservation_index.csv` (all 8 rows) must be reproduced to 1e-9 by notebook 43's own function,
and by this notebook's estimator, before anything new is computed.

**Part A · Agreement by gradient strength.**
- *Strength is cross-fitted.* Binning on one dataset's |gradient| selects that dataset's noise
  extremes (regression to the mean). The external references' donors are split into two folds
  (sha256 order, alternating: Lake healthy cortex 4/3, Census male mouse snRNA 6/6). Strength
  comes from fold A: the percentile ranks of |Lake_A| and |snRNA_A| gradients, combined by their
  **maximum** (strong in either species; the maximum does not presuppose zonation in both, which
  would favour shared genes). Everything is evaluated on fold B and our data, then the folds swap
  and the two directions are averaged.
- *Per bin*, notebook 43's index on the bin's genes: cross r*(our human, our mouse), ceilings
  r*(our human, Lake_B) and r*(our mouse, snRNA_B), the external cross r*(Lake_B, snRNA_B), and both
  indices. Gradients are winsorised at q = 0.01 within the bin and reliability uses the same
  winsorised values. An r* is not estimable if a reliability in the bin is < 0.2.
- *Bins:* strength quantiles 0–50, 50–80, 80–90, 90–95, 95–99, 99–100%. **BOTTOM** = below the 80th
  percentile, **TOP** = at or above the 95th.
- *Intervals:* 1,000 gene bootstraps and 1,000 module-block bootstraps (blocks: co-expression
  modules, Part B). A bound is not estimable if > 10% of draws are.
- *Secondary schemes:* human rank only, mouse rank only, mean rank, the referee's same-data
  binning on |our mouse gradient|, and the external pair binned on our strength with ceilings that
  do not involve our data.
- *Direction agreement* among genes zonated in both species (notebook 43 symmetric classes) by
  strength bin.

**Decision rules.**
- (i) *Concentration:* in S2 − S1 and S3 − early, index(TOP) − index(BOTTOM) ≥ 0.25 with a 95%
  interval excluding 0, under gene and module-block bootstraps (cross r* if index(BOTTOM) is not
  estimable).
- (ii) *Reproducible but not shared:* in BOTTOM both ceilings have lower bounds ≥ 0.3 and the
  index has an upper bound < 0.5. *Not reliably zonated:* a BOTTOM ceiling with upper bound < 0.3
  or not estimable.
- Verdict: (i) + (ii) → title supported; (i) + not reliably zonated → "agreement is confined to
  genes that are reliably zonated, and few genes are"; (i) + in between → undecided for weak
  genes; (i) fails → drop the strength framing.

**Part B · Index sensitivity.** Winsorisation q ∈ {0, 0.01, 0.05} × reliability {notebook 43,
matched}; gene, module-block (k = 100; 50 and 200), external-donor and donor + gene bootstraps;
the S2 − S1 human ceiling's dependence on one dataset pair. "Largely different genes" stands if
every upper bound of the primary index for S2 − S1 and S3 − early is < 0.5.

## 1 · Folders and settings

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
reference43 = results_root / 'pt_revision_classes' / 'reviewed' / 'tables'
external = data_root / 'external'
output = results_root / 'pt_conservation_strength'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
census = external / 'census_pt_segments'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            reference43 / 'conservation_index.csv', reference43 / 'gene_classes_symmetric.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            census / 'mouse_pt_segment_pseudobulk_counts.csv.gz', census / 'mouse_pt_segment_pseudobulk_meta.csv',
            census / 'human_pt_segment_pseudobulk_counts.csv.gz', census / 'human_pt_segment_pseudobulk_meta.csv', lake_file]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '49.conservation_strength.1'
SEED43, SEED, N_BOOT = 43, 49, 1000                       # notebook 43's gene-bootstrap seed; this notebook's seed
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.      # notebook 43
Q_PUBLISHED, Q_GRID = .01, (0., .01, .05)
EDGES = (0., .5, .8, .9, .95, .99, 1.)
MIN_RELIABILITY, CEILING_MIN, INDEX_STRONG, CONCENTRATION_MIN, MAX_MISSING = .2, .3, .5, .25, .1
N_COMPONENTS, N_MODULES, N_MODULES_SENSITIVITY = 50, 100, (50, 200)
print('Results folder:', output)

## 2 · Inputs (notebook 43, sections 2–4, reviewed labels)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import re
from scipy import sparse
from scipy.spatial import cKDTree
from IPython.display import display

import pseudospace.conservation_strength as strength_module
import pseudospace.zonation_amplitude as amplitude_module
import pseudospace.zonation_reliability as reliability_module
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.repeated_inputs import _read_and_validate_segmentation
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.zonation_amplitude import group_sums, log_ratio
from pseudospace.zonation_reliability import conservation_index, paired_gradient, unpaired_gradient

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(amplitude_module.__file__), Path(reliability_module.__file__),
               Path(strength_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
measured = adata.var.measured_in_both_inputs.to_numpy()
gene_space = adata.var_names[measured]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[:, measured]
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
library = np.asarray(counts.sum(axis=1)).ravel()


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']
cortical_s3 = np.zeros(len(specimen), bool)
for name in expected['mouse']:
    limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
    cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
segment_region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)

# Specimen x segment pseudobulks and gradients (notebook 43, sections 2-3).
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}
PARTS = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}
pb_names, pb_sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), segment_region.astype(str)))
pb_frame = pd.DataFrame(pb_sums, index=pb_names, columns=gene_space)
ours_counts = pd.DataFrame({(n, label): pb_frame.loc[[f'{n}|{s}' for s in members if f'{n}|{s}' in pb_frame.index]].sum()
                            for n in names for label, members in PARTS.items()})
ours_library = pd.Series({(n, label): float(library[(specimen == n) & np.isin(segment_region, members)].sum())
                          for n in names for label, members in PARTS.items()})
ours = pd.DataFrame({(contrast, n): log_ratio(ours_counts[(n, late)], ours_library[(n, late)], ours_counts[(n, early)], ours_library[(n, early)])
                     for contrast, (late, early) in CONTRASTS.items() for n in names}, index=gene_space)


def count_filter(specimens, parts):
    return ours_counts[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)


print(pd.crosstab(specimen, segment_region))

### External references (notebook 43, section 4)

In [ ]:
def microdissection_male():
    table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
    table = table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
    per_mouse = {}
    for column in table.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    values = {key: np.log2(table[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
              for key, cols in per_mouse.items()}
    return pd.DataFrame(values).reindex(gene_space)


def census_segments(label, segment_map):
    table = pd.read_csv(census / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(census / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    counts_out, totals, info = {}, {}, {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        for seg_name in needed:
            index = block.index[block.segment.eq(seg_name)]
            counts_out[(donor, seg_name)] = table.loc[index].sum()
            totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
        info[donor] = sex
    return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)


def lake_pseudobulk():
    """Lake/KPMP cortex PT nuclei summed per donor x disease x segment (notebook 43's function)."""
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=gene_space, fill_value=0.)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy()}


micro_male = microdissection_male()
sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'})
census_h_counts, census_h_totals, census_h_sex = census_segments('human', {
    'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'})
lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': list(gene_space),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake['nuclei'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=gene_space)
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
lake_counts = pd.DataFrame({(d, s): lake_sums.loc[lake_meta.index[lake_meta.donor.eq(d) & lake_meta.segment.eq(s)]].sum()
                            for d in lake_donors for s in ('S1', 'S2', 'S3')})
lake_totals = pd.Series({(d, s): float(lake_meta.loc[lake_meta.donor.eq(d) & lake_meta.segment.eq(s), 'total'].sum())
                         for d in lake_donors for s in ('S1', 'S2', 'S3')})
healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']
male_sn = list(sn_sex.index[sn_sex.eq('male')])
print(f'Lake healthy cortex donors: {len(healthy_lake)}; Census male mouse snRNA donors: {len(male_sn)}; '
      f'Census human donors: {len(census_h_sex)}')


def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def donor_contrasts(count_frame, totals, donors, contrast):
    """Per-donor log2 CPM contrast and per-segment level (notebook 43); early pools S1 + S2 counts."""
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T

### Gradients with sampling noise (notebook 43, section 5)

In [ ]:
def ours_gradient(contrast, specimens):
    late, early = CONTRASTS[contrast]
    expressed = set(gene_space[count_filter(specimens, [late, early]).to_numpy()])
    return paired_gradient(ours[[(contrast, n) for n in specimens]].T.reset_index(drop=True)), expressed


def donor_gradient(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
    return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])


def census_human_gradient():
    donors = census_h_sex.index
    late = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}).T
    early = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}).T
    keep = np.maximum(late.mean(), early.mean()).ge(REFERENCE_LEVEL) & (late - early).notna().all()
    return paired_gradient((late - early).loc[:, keep]), set(keep.index[keep.to_numpy()])


def segment_replicates_gradient(by_segment, contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    if early == 'early':
        parts = [by_segment['S1'], by_segment['S2']]
        mean = by_segment[late].mean() - (parts[0].mean() + parts[1].mean()) / 2
        noise = by_segment[late].var(ddof=1) / len(by_segment[late]) + sum(p.var(ddof=1) / len(p) for p in parts) / 4
        level = pd.concat([by_segment[late].mean(), (parts[0].mean() + parts[1].mean()) / 2], axis=1).max(axis=1)
    else:
        mean, noise = unpaired_gradient(by_segment[late], by_segment[early])
        level = pd.concat([by_segment[late].mean(), by_segment[early].mean()], axis=1).max(axis=1)
    keep = level.ge(REFERENCE_LEVEL) & mean.notna()
    return (mean, noise), set(keep.index[keep.to_numpy()])


micro_male_segments = {s: micro_male[[c for c in micro_male.columns if c[0] == s]].T.reset_index(drop=True) for s in ('S1', 'S2', 'S3')}


def reference_contrast(contrast):
    return 'S3-early' if contrast == 'S3c-early' else contrast


def datasets_for(contrast):
    rc = reference_contrast(contrast)
    out = {('human', 'ours'): ours_gradient(contrast, expected['human']),
           ('mouse', 'ours'): ours_gradient(contrast, expected['mouse']),
           ('human', 'Lake healthy'): donor_gradient(lake_counts, lake_totals, healthy_lake, rc),
           ('mouse', 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, rc),
           ('mouse', 'micro male GSE150338'): segment_replicates_gradient(micro_male_segments, rc)}
    if contrast != 'S2-S1':
        out[('human', 'Census human')] = census_human_gradient()
    return out


datasets = {contrast: datasets_for(contrast) for contrast in ('S2-S1', 'S3-early', 'S3c-early')}
index_genes = {c: pd.Index(sorted(d[('human', 'ours')][1] & d[('mouse', 'ours')][1] & d[('human', 'Lake healthy')][1]
                                  & d[('mouse', 'snRNA male')][1])) for c, d in datasets.items()}
print({c: len(g) for c, g in index_genes.items()}, 'genes in the index set (expressed in all four datasets)')

## 3 · Guard: reproduce notebook 43's published conservation index exactly

Validation cell (not cached): notebook 43's function at its published settings, then this
notebook's estimator (`conservation_strength.subset_statistics`) with notebook 43's reliability
convention and the same gene draws.

In [ ]:
from pseudospace.conservation_strength import (STATS, binned_statistics, bin_labels, block_draws, coexpression_modules,
                                               donor_folds, gene_draws, percentile_interval, percentile_rank,
                                               resampled_gradient, strength_bins, strength_score, subset_statistics)

published = pd.read_csv(reference43 / 'conservation_index.csv')
VERSIONS = {'primary: ours human vs ours mouse': (('human', 'ours'), ('mouse', 'ours'), ('human', 'Lake healthy'), ('mouse', 'snRNA male')),
            'external: Lake vs mouse snRNA male': (('human', 'Lake healthy'), ('mouse', 'snRNA male'), ('human', 'ours'), ('mouse', 'ours')),
            'external, alternative ceilings: Lake vs snRNA male (Census human, microdissection)':
                (('human', 'Lake healthy'), ('mouse', 'snRNA male'), ('human', 'Census human'), ('mouse', 'micro male GSE150338'))}
FIELDS = ('index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human', 'ceiling_mouse', 'genes')
reproduced = []
for row in published.itertuples(index=False):
    data = datasets[row.contrast]
    keys = VERSIONS[row.version]
    genes = (index_genes[row.contrast] if not row.version.startswith('external, alternative')
             else pd.Index(sorted(set.intersection(*[data[k][1] for k in keys]))))
    result = conservation_index(*[data[k][0] for k in keys], genes, n_boot=N_BOOT, seed=SEED43)
    for field in FIELDS:
        if not np.isclose(result[field], getattr(row, field), rtol=0, atol=1e-9):
            raise ValueError(f'Notebook 43 not reproduced: {row.contrast} · {row.version} · {field}')
    reproduced.append({'contrast': row.contrast, 'version': row.version, **result})
print(f'Notebook 43 reproduced exactly: {len(reproduced)} of {len(published)} rows.')


def as_arrays(gradient, genes):
    mean, noise = gradient
    return mean.reindex(genes).to_numpy(float), noise.reindex(genes).to_numpy(float)


def full_data(contrast, genes):
    d = datasets[contrast]
    return {'human': as_arrays(d[('human', 'ours')][0], genes), 'mouse': as_arrays(d[('mouse', 'ours')][0], genes),
            'human_ref': as_arrays(d[('human', 'Lake healthy')][0], genes), 'mouse_ref': as_arrays(d[('mouse', 'snRNA male')][0], genes)}


for row in published[published.version.str.startswith(('primary', 'external: '))].itertuples(index=False):
    data = full_data(row.contrast, index_genes[row.contrast])
    stat = 'index' if row.version.startswith('primary') else 'index_external'
    n = len(index_genes[row.contrast])
    point = subset_statistics(data, np.arange(n), q=Q_PUBLISHED, match_reliability=False)
    draws = np.array([subset_statistics(data, idx, q=Q_PUBLISHED, match_reliability=False)[stat] for idx in gene_draws(n, N_BOOT, SEED43)])
    low, high, _ = percentile_interval(draws[:, None])
    for value, target in ((point[stat], row.index), (low[0], row.ci_low), (high[0], row.ci_high),
                          (point['rstar_ceiling_human'], row.ceiling_human), (point['rstar_ceiling_mouse'], row.ceiling_mouse)):
        if not np.isclose(value, target, rtol=0, atol=1e-9):
            raise ValueError(f'New estimator does not reproduce notebook 43: {row.contrast} · {row.version}')
print('The new estimator reproduces notebook 43 (point, interval and ceilings) for the primary and external indices.')
display(pd.DataFrame(reproduced)[['contrast', 'version', *FIELDS]].round(3))

## 4 · Co-expression modules for the module-block bootstrap

Blocks are co-expression modules of our PT structures (index genes, union over contrasts):
log-normalised expression centred per specimen, scaled per gene within species, both species
weighted equally; 50-component truncated SVD; gene loadings clustered by k-means. Genes that
co-vary across structures share zonation programs, cell state and spill-over, so their gradient
signals and errors are dependent; a gene bootstrap treats them as independent. Pathway annotations
cover a minority of genes and overlap, so they would not give a partition of the gene set.

In [ ]:
module_genes = pd.Index(sorted(set().union(*[set(g) for g in index_genes.values()])))
species_of_structure = np.where(np.isin(specimen, expected['mouse']), 'mouse', 'human')


def compute_modules():
    matrix = lognorm[:, gene_space.get_indexer(module_genes)]
    return {f'k{k}': coexpression_modules(matrix, specimen, species_of_structure, n_components=N_COMPONENTS, n_modules=k, seed=SEED)
            for k in (N_MODULES, *N_MODULES_SENSITIVITY)}


modules = cached_payload('coexpression_modules', compute_modules, root=cache,
                         params={'components': N_COMPONENTS, 'k': [N_MODULES, *N_MODULES_SENSITIVITY], 'seed': SEED},
                         inputs={'genes': list(module_genes), 'obs': list(adata.obs_names), 'lognorm': digest(lognorm)}, code=code)
module_of = {k: pd.Series(v, index=module_genes) for k, v in modules.items()}
module_sizes = pd.DataFrame({k: v.value_counts().describe()[['count', 'min', '50%', 'max']] for k, v in module_of.items()}).T
display(module_sizes.rename(columns={'count': 'modules', '50%': 'median genes'}))

## 5 · Part A: agreement by gradient strength

Strength is cross-fitted on external donor folds (section 0 of the protocol). For each contrast and
scheme the cached stage stores the point estimates and the 1,000 gene (and, for the primary
scheme, module-block) bootstrap draws of every statistic in every bin and decision group.

In [ ]:
BINS = bin_labels(EDGES)
SUBSETS = {**{label: [i] for i, label in enumerate(BINS)}, 'BOTTOM (<80%)': [0, 1], 'TOP (≥95%)': [4, 5],
           'all genes': list(range(len(BINS)))}
SUBSET_NAMES = list(SUBSETS)
SCHEMES = {'max rank (primary)': 'max', 'mean rank': 'mean', 'human rank only': 'human', 'mouse rank only': 'mouse'}
lake_fold = pd.Series(donor_folds(healthy_lake), index=healthy_lake)
sn_fold = pd.Series(donor_folds(male_sn), index=male_sn)
print('Lake folds:', lake_fold.value_counts().sort_index().to_dict(), '· snRNA male folds:', sn_fold.value_counts().sort_index().to_dict())


def reference_per_donor(contrast, genes):
    rc = reference_contrast(contrast)
    lake_pd = donor_contrasts(lake_counts, lake_totals, healthy_lake, rc)[0][genes]
    sn_pd = donor_contrasts(sn_counts, sn_totals, male_sn, rc)[0][genes]
    return lake_pd, sn_pd


def fold_gradient(per_donor, folds, fold):
    mean, noise = paired_gradient(per_donor.loc[folds.index[folds.eq(fold)]])
    return mean.to_numpy(float), noise.to_numpy(float)


def crossfit_directions(contrast, how):
    """[(evaluation data, bins)] for the two directions: strength from fold f, evaluation on fold 1 - f."""
    genes = index_genes[contrast]
    lake_pd, sn_pd = reference_per_donor(contrast, genes)
    ours_data = full_data(contrast, genes)
    out = []
    for f in (0, 1):
        rank_h = percentile_rank(fold_gradient(lake_pd, lake_fold, f)[0])
        rank_m = percentile_rank(fold_gradient(sn_pd, sn_fold, f)[0])
        score = {'human': rank_h, 'mouse': rank_m}.get(how)
        score = strength_score([rank_h, rank_m], how) if score is None else score
        data = {'human': ours_data['human'], 'mouse': ours_data['mouse'],
                'human_ref': fold_gradient(lake_pd, lake_fold, 1 - f), 'mouse_ref': fold_gradient(sn_pd, sn_fold, 1 - f)}
        out.append((data, strength_bins(score, EDGES)))
    return out


def naive_directions(contrast):
    """The referee's scheme: bins on |our mouse gradient| (the same data), full references."""
    data = full_data(contrast, index_genes[contrast])
    return [(data, strength_bins(percentile_rank(data['mouse'][0]), EDGES))]


def run_bootstrap(directions, draws, q):
    return np.stack([binned_statistics(directions, idx, SUBSETS, q=q, min_reliability=MIN_RELIABILITY) for idx in draws])


def strength_stage(contrast):
    genes = index_genes[contrast]
    n = len(genes)
    blocks = module_of[f'k{N_MODULES}'].reindex(genes).to_numpy()
    out = {}
    schemes = {**{name: crossfit_directions(contrast, how) for name, how in SCHEMES.items()},
               'same-data |our mouse| (referee)': naive_directions(contrast)}
    for name, directions in schemes.items():
        for q in ((Q_PUBLISHED, 0.) if name == 'max rank (primary)' else (Q_PUBLISHED,)):
            out[f'{name}|q{q}|point'] = binned_statistics(directions, np.arange(n), SUBSETS, q=q, min_reliability=MIN_RELIABILITY)
            out[f'{name}|q{q}|gene'] = run_bootstrap(directions, gene_draws(n, N_BOOT, SEED), q)
        if name == 'max rank (primary)':
            out[f'{name}|q{Q_PUBLISHED}|module'] = run_bootstrap(directions, block_draws(blocks, N_BOOT, SEED), Q_PUBLISHED)
            for f in (0, 1):
                out[f'{name}|q{Q_PUBLISHED}|direction{f}'] = binned_statistics([directions[f]], np.arange(n), SUBSETS, q=Q_PUBLISHED,
                                                                              min_reliability=MIN_RELIABILITY)
    return out


strength_results = {}
for contrast in index_genes:
    genes_ = index_genes[contrast]
    strength_results[contrast] = cached_payload(
        f'strength_{contrast}', lambda c=contrast: strength_stage(c), root=cache,
        params={'edges': EDGES, 'subsets': SUBSETS, 'schemes': SCHEMES, 'q': [Q_PUBLISHED, 0.], 'n_boot': N_BOOT, 'seed': SEED,
                'min_reliability': MIN_RELIABILITY, 'k': N_MODULES},
        inputs={'genes': list(genes_), 'ours': digest(ours), 'lake': digest(lake_counts), 'sn': digest(sn_counts.sort_index(axis=1)),
                'folds': [lake_fold.to_dict(), sn_fold.to_dict()], 'modules': digest(module_of[f'k{N_MODULES}'])}, code=code)

### Strength curves (tables)

In [ ]:
def summarise(stage, key_prefix, boot):
    point = stage[f'{key_prefix}|point']
    low, high, missing = percentile_interval(stage[f'{key_prefix}|{boot}'], max_missing=MAX_MISSING)
    rows = []
    for i, subset in enumerate(SUBSET_NAMES):
        for j, stat in enumerate(STATS):
            rows.append({'subset': subset, 'statistic': stat, 'value': point[i, j], 'ci_low': low[i, j], 'ci_high': high[i, j],
                         'missing_draws': missing[i, j]})
    return pd.DataFrame(rows)


curve_rows = []
for contrast, stage in strength_results.items():
    for key in sorted({k.rsplit('|', 1)[0] for k in stage if k.endswith('|point')}):
        scheme, q = key.rsplit('|', 1)
        for boot in ('gene', 'module'):
            if f'{key}|{boot}' in stage:
                curve_rows.append(summarise(stage, key, boot).assign(contrast=contrast, scheme=scheme, q=float(q[1:]), bootstrap=boot))
curves = pd.concat(curve_rows, ignore_index=True)[['contrast', 'scheme', 'q', 'bootstrap', 'subset', 'statistic', 'value', 'ci_low', 'ci_high', 'missing_draws']]
curves.to_csv(tables / 'strength_curves.csv', index=False)
direction_rows = []
for contrast, stage in strength_results.items():
    for f in (0, 1):
        frame = pd.DataFrame(stage[f'max rank (primary)|q{Q_PUBLISHED}|direction{f}'], index=SUBSET_NAMES, columns=STATS)
        direction_rows.append(frame.assign(contrast=contrast, direction=f'strength fold {f} → evaluation fold {1 - f}'))
pd.concat(direction_rows).rename_axis('subset').reset_index().to_csv(tables / 'strength_curves_by_direction.csv', index=False)


def wide(contrast, scheme='max rank (primary)', q=Q_PUBLISHED, boot='gene', stats=('genes', 'rstar_cross', 'rstar_ceiling_human',
                                                                                   'rstar_ceiling_mouse', 'index', 'index_external')):
    block = curves[curves.contrast.eq(contrast) & curves.scheme.eq(scheme) & curves.q.eq(q) & curves.bootstrap.eq(boot)
                   & curves.statistic.isin(stats)]
    table = block.assign(text=[f'{v:.2f} ({lo:.2f}, {hi:.2f})' if s != 'genes' else f'{v:.0f}'
                               for v, lo, hi, s in zip(block.value, block.ci_low, block.ci_high, block.statistic)])
    return table.pivot(index='subset', columns='statistic', values='text').reindex(SUBSET_NAMES)[list(stats)]


for contrast in index_genes:
    print(f'\n{contrast} · primary scheme (cross-fitted max rank), q = 0.01, gene bootstrap')
    display(wide(contrast))

### Decision rules (i) and (ii)

In [ ]:
def group_draws(stage, key, boot, subset, stat):
    return stage[f'{key}|{boot}'][:, SUBSET_NAMES.index(subset), STATS.index(stat)]


def group_point(stage, key, subset, stat):
    return stage[f'{key}|point'][SUBSET_NAMES.index(subset), STATS.index(stat)]


PRIMARY_KEY = f'max rank (primary)|q{Q_PUBLISHED}'
decision_rows = []
for contrast, stage in strength_results.items():
    bottom_estimable = np.isfinite(group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', 'index'))
    stat = 'index' if bottom_estimable else 'rstar_cross'
    for boot in ('gene', 'module'):
        diff = group_draws(stage, PRIMARY_KEY, boot, 'TOP (≥95%)', stat) - group_draws(stage, PRIMARY_KEY, boot, 'BOTTOM (<80%)', stat)
        d_low, d_high, d_missing = percentile_interval(diff[:, None], max_missing=MAX_MISSING)
        bounds = {}
        for s in ('rstar_ceiling_human', 'rstar_ceiling_mouse', 'index', 'rstar_cross'):
            lo, hi, _ = percentile_interval(group_draws(stage, PRIMARY_KEY, boot, 'BOTTOM (<80%)', s)[:, None], max_missing=MAX_MISSING)
            bounds[s] = (group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', s), lo[0], hi[0])
        point_diff = group_point(stage, PRIMARY_KEY, 'TOP (≥95%)', stat) - group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', stat)
        concentration = bool(point_diff >= CONCENTRATION_MIN and d_low[0] > 0)
        ceilings_low = [bounds[s][1] for s in ('rstar_ceiling_human', 'rstar_ceiling_mouse')]
        ceilings_high = [bounds[s][2] for s in ('rstar_ceiling_human', 'rstar_ceiling_mouse')]
        reproducible_not_shared = bool(all(np.isfinite(ceilings_low)) and min(ceilings_low) >= CEILING_MIN
                                       and np.isfinite(bounds['index'][2]) and bounds['index'][2] < INDEX_STRONG)
        not_reliably_zonated = bool(any(not np.isfinite(v) for v in ceilings_high) or min(ceilings_high) < CEILING_MIN)
        decision_rows.append({'contrast': contrast, 'bootstrap': boot, 'statistic compared': stat,
                              'TOP': group_point(stage, PRIMARY_KEY, 'TOP (≥95%)', stat),
                              'BOTTOM': group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', stat),
                              'TOP − BOTTOM': point_diff, 'difference low': d_low[0], 'difference high': d_high[0],
                              '(i) concentration': concentration,
                              'BOTTOM human ceiling': '{:.2f} ({:.2f}, {:.2f})'.format(*bounds['rstar_ceiling_human']),
                              'BOTTOM mouse ceiling': '{:.2f} ({:.2f}, {:.2f})'.format(*bounds['rstar_ceiling_mouse']),
                              'BOTTOM index': '{:.2f} ({:.2f}, {:.2f})'.format(*bounds['index']),
                              '(ii) reproducible but not shared': reproducible_not_shared,
                              'BOTTOM not reliably zonated': not_reliably_zonated})
strength_decisions = pd.DataFrame(decision_rows)
strength_decisions.to_csv(tables / 'strength_decision_rules.csv', index=False)
display(strength_decisions)

primary_contrasts = strength_decisions[strength_decisions.contrast.isin(['S2-S1', 'S3-early'])]
RULE_I = bool(primary_contrasts['(i) concentration'].all())
RULE_II = bool(primary_contrasts['(ii) reproducible but not shared'].all())
NOT_ZONATED = bool(primary_contrasts['BOTTOM not reliably zonated'].any())
if RULE_I and RULE_II:
    TITLE_VERDICT = 'supported: shared zonation concentrates in the strongest genes; reproducible weaker gradients differ'
elif RULE_I and NOT_ZONATED:
    TITLE_VERDICT = 'reword: agreement is confined to genes that are reliably zonated, and few genes are'
elif RULE_I:
    TITLE_VERDICT = 'partly: agreement concentrates in the strongest genes; weak-gene ceilings are too uncertain to call species difference'
else:
    TITLE_VERDICT = 'not supported: no strength dependence; drop the strongest-genes framing'
print('Rule (i):', RULE_I, '· rule (ii):', RULE_II, '· BOTTOM not reliably zonated:', NOT_ZONATED)
print('Title verdict:', TITLE_VERDICT)

### Bin-level classification and gene counts

Each of the six bins is classed with the rules of (ii) (gene bootstrap; the module-block result
is shown alongside): **reproducible, not shared**, **not reliably zonated**, **shared** (index
lower bound ≥ 0.5) or **uncertain**.

In [ ]:
def bin_class(stage, key, boot, subset):
    lo_h, hi_h, _ = percentile_interval(group_draws(stage, key, boot, subset, 'rstar_ceiling_human')[:, None], max_missing=MAX_MISSING)
    lo_m, hi_m, _ = percentile_interval(group_draws(stage, key, boot, subset, 'rstar_ceiling_mouse')[:, None], max_missing=MAX_MISSING)
    lo_i, hi_i, _ = percentile_interval(group_draws(stage, key, boot, subset, 'index')[:, None], max_missing=MAX_MISSING)
    if not (np.isfinite(hi_h[0]) and np.isfinite(hi_m[0])) or min(hi_h[0], hi_m[0]) < CEILING_MIN:
        return 'not reliably zonated'
    if min(lo_h[0], lo_m[0]) >= CEILING_MIN and np.isfinite(hi_i[0]) and hi_i[0] < INDEX_STRONG:
        return 'reproducible, not shared'
    if np.isfinite(lo_i[0]) and lo_i[0] >= INDEX_STRONG:
        return 'shared'
    return 'uncertain'


bin_rows = []
for contrast, stage in strength_results.items():
    for subset in BINS:
        bin_rows.append({'contrast': contrast, 'bin': subset, 'genes': int(group_point(stage, PRIMARY_KEY, subset, 'genes')),
                         'class (gene bootstrap)': bin_class(stage, PRIMARY_KEY, 'gene', subset),
                         'class (module bootstrap)': bin_class(stage, PRIMARY_KEY, 'module', subset)})
bin_classes = pd.DataFrame(bin_rows)
bin_classes.to_csv(tables / 'strength_bin_classes.csv', index=False)
display(bin_classes.pivot(index='bin', columns='contrast', values='class (gene bootstrap)').reindex(BINS))
gene_counts = bin_classes.pivot_table(index='class (gene bootstrap)', columns='contrast', values='genes', aggfunc='sum', fill_value=0)
display(gene_counts)

### Secondary schemes: asymmetry and the size of the selection effect

In [ ]:
scheme_rows = []
for contrast in index_genes:
    for scheme in (*SCHEMES, 'same-data |our mouse| (referee)'):
        block = curves[curves.contrast.eq(contrast) & curves.scheme.eq(scheme) & curves.q.eq(Q_PUBLISHED) & curves.bootstrap.eq('gene')]
        for subset in ('BOTTOM (<80%)', 'TOP (≥95%)', BINS[-1]):
            values = block[block.subset.eq(subset)].set_index('statistic')
            scheme_rows.append({'contrast': contrast, 'scheme': scheme, 'subset': subset,
                                'raw r': values.loc['r_cross', 'value'], 'r*': values.loc['rstar_cross', 'value'],
                                'human ceiling': values.loc['rstar_ceiling_human', 'value'],
                                'mouse ceiling': values.loc['rstar_ceiling_mouse', 'value'],
                                'index': values.loc['index', 'value'],
                                'index CI': f"({values.loc['index', 'ci_low']:.2f}, {values.loc['index', 'ci_high']:.2f})",
                                'sign agreement': values.loc['sign_agreement', 'value']})
scheme_table = pd.DataFrame(scheme_rows)
scheme_table.to_csv(tables / 'strength_schemes.csv', index=False)
display(scheme_table.round(3))

### Direction agreement among genes zonated in both species

Notebook 43's symmetric classes (probe-balanced genes): among genes zonated in both species
(conserved or reversal), the share conserved, by strength bin. Strength here is the maximum rank
on the full references; sign agreement between our species is not biased by selecting on
|reference|.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

classes43 = pd.read_csv(reference43 / 'gene_classes_symmetric.csv', index_col=0)
agreement_rows = []
for contrast in index_genes:
    genes = index_genes[contrast]
    data = full_data(contrast, genes)
    full_bins = pd.Series(strength_bins(strength_score([percentile_rank(data['human_ref'][0]), percentile_rank(data['mouse_ref'][0])]), EDGES),
                          index=genes)
    klass = classes43.loc[classes43.probe_balanced.astype(bool), contrast].dropna()
    both = klass[klass.isin(['conserved', 'reversal'])]
    missing = int((~both.index.isin(genes)).sum())
    both = both[both.index.isin(genes)]
    sign = pd.Series(np.sign(data['human'][0]) == np.sign(data['mouse'][0]), index=genes)
    for b, label in enumerate([*BINS, 'all genes']):
        members = genes if label == 'all genes' else genes[full_bins.eq(b).to_numpy()]
        zb = both[both.index.isin(members)]
        n_both, n_conserved = len(zb), int(zb.eq('conserved').sum())
        low, high = proportion_confint(n_conserved, n_both, method='wilson') if n_both else (np.nan, np.nan)
        agreement_rows.append({'contrast': contrast, 'bin': label, 'genes': len(members), 'zonated in both': n_both,
                               'conserved': n_conserved, 'share conserved': n_conserved / n_both if n_both else np.nan,
                               'share low': low, 'share high': high, 'raw sign agreement (all genes)': float(sign.loc[members].mean()),
                               'zonated-in-both genes outside the index set': missing})
direction_agreement = pd.DataFrame(agreement_rows)
direction_agreement.to_csv(tables / 'direction_agreement_by_strength.csv', index=False)
display(direction_agreement.round(3))

### External pair binned on our strength

The reverse cross-fit: strength from our data (maximum rank of |our human| and |our mouse|), and
agreement between Lake and Census male mouse snRNA, with ceilings that do not involve our data:
snRNA male against male microdissection (mouse), and Lake against Census human (human; S3
contrasts only, since Census human has no S2).

In [ ]:
def external_stage():
    out = {}
    for contrast in index_genes:
        d = datasets[contrast]
        keys = [('human', 'ours'), ('mouse', 'ours'), ('human', 'Lake healthy'), ('mouse', 'snRNA male'), ('mouse', 'micro male GSE150338')]
        keys += [('human', 'Census human')] if contrast != 'S2-S1' else []
        genes = pd.Index(sorted(set.intersection(*[d[k][1] for k in keys])))
        human_ref = d[('human', 'Census human')][0] if contrast != 'S2-S1' else d[('human', 'Lake healthy')][0]
        data = {'human': as_arrays(d[('human', 'Lake healthy')][0], genes), 'mouse': as_arrays(d[('mouse', 'snRNA male')][0], genes),
                'human_ref': as_arrays(human_ref, genes), 'mouse_ref': as_arrays(d[('mouse', 'micro male GSE150338')][0], genes)}
        score = strength_score([percentile_rank(as_arrays(d[('human', 'ours')][0], genes)[0]),
                                percentile_rank(as_arrays(d[('mouse', 'ours')][0], genes)[0])])
        directions = [(data, strength_bins(score, EDGES))]
        out[f'{contrast}|point'] = binned_statistics(directions, np.arange(len(genes)), SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
        out[f'{contrast}|gene'] = run_bootstrap(directions, gene_draws(len(genes), N_BOOT, SEED), Q_PUBLISHED)
    return out


external_results = cached_payload('external_pair_by_our_strength', external_stage, root=cache,
                                  params={'edges': EDGES, 'subsets': SUBSETS, 'q': Q_PUBLISHED, 'n_boot': N_BOOT, 'seed': SEED,
                                          'min_reliability': MIN_RELIABILITY},
                                  inputs={'ours': digest(ours), 'lake': digest(lake_counts), 'sn': digest(sn_counts.sort_index(axis=1)),
                                          'micro': digest(micro_male), 'census_human': digest(census_h_counts.sort_index(axis=1))}, code=code)
external_rows = []
for contrast in index_genes:
    frame = summarise(external_results, contrast, 'gene').assign(contrast=contrast)
    if contrast == 'S2-S1':      # no independent human reference: the human ceiling and index are undefined
        undefined = frame.statistic.isin(['r_ceiling_human', 'rstar_ceiling_human', 'index', 'index_external', 'r_external',
                                          'rstar_external', 'reliability_human_ref'])
        frame.loc[undefined, ['value', 'ci_low', 'ci_high']] = np.nan
    external_rows.append(frame)
external_curves = pd.concat(external_rows, ignore_index=True)
external_curves['statistic'] = external_curves.statistic.replace({
    'rstar_cross': 'rstar Lake vs snRNA male', 'r_cross': 'r Lake vs snRNA male', 'rstar_ceiling_human': 'rstar Lake vs Census human',
    'rstar_ceiling_mouse': 'rstar snRNA male vs microdissection', 'index': 'index (external pair, independent ceilings)'})
external_curves.to_csv(tables / 'external_pair_by_our_strength.csv', index=False)
show = external_curves[external_curves.statistic.isin(['rstar Lake vs snRNA male', 'rstar Lake vs Census human',
                                                       'rstar snRNA male vs microdissection', 'index (external pair, independent ceilings)'])]
display(show.assign(text=[f'{v:.2f} ({lo:.2f}, {hi:.2f})' for v, lo, hi in zip(show.value, show.ci_low, show.ci_high)])
        .pivot_table(index=['contrast', 'subset'], columns='statistic', values='text', aggfunc='first')
        .reindex(pd.MultiIndex.from_product([list(index_genes), SUBSET_NAMES])))

### Post-hoc diagnostic (not pre-registered): why the S2 − S1 weak-gene human ceiling is missing

Added after seeing that the S2 − S1 human ceiling is not estimable below the 80th percentile: the
Lake fold of 3–4 donors has reliability < 0.2 there. Here strength is the mouse rank in snRNA
fold A only, so the **full** Lake reference (all healthy donors, never used for strength) gives the
human ceiling without selection bias; the mouse ceiling uses snRNA fold B. This diagnostic does not
change the pre-registered verdict.

In [ ]:
def mouse_strength_full_lake(contrast):
    genes = index_genes[contrast]
    _, sn_pd = reference_per_donor(contrast, genes)
    ours_data = full_data(contrast, genes)
    return [({**ours_data, 'mouse_ref': fold_gradient(sn_pd, sn_fold, 1 - f)},
             strength_bins(percentile_rank(fold_gradient(sn_pd, sn_fold, f)[0]), EDGES)) for f in (0, 1)]


def posthoc_stage():
    out = {}
    for contrast in index_genes:
        directions = mouse_strength_full_lake(contrast)
        n = len(index_genes[contrast])
        out[f'{contrast}|point'] = binned_statistics(directions, np.arange(n), SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
        out[f'{contrast}|gene'] = run_bootstrap(directions, gene_draws(n, N_BOOT, SEED), Q_PUBLISHED)
    return out


posthoc = cached_payload('posthoc_mouse_strength_full_lake', posthoc_stage, root=cache,
                         params={'edges': EDGES, 'subsets': SUBSETS, 'q': Q_PUBLISHED, 'n_boot': N_BOOT, 'seed': SEED,
                                 'min_reliability': MIN_RELIABILITY},
                         inputs={'ours': digest(ours), 'lake': digest(lake_counts), 'sn': digest(sn_counts.sort_index(axis=1)),
                                 'folds': sn_fold.to_dict()}, code=code)
posthoc_table = pd.concat([summarise(posthoc, contrast, 'gene').assign(contrast=contrast) for contrast in index_genes], ignore_index=True)
posthoc_table.to_csv(tables / 'posthoc_mouse_strength_full_lake.csv', index=False)
show = posthoc_table[posthoc_table.statistic.isin(['rstar_cross', 'rstar_ceiling_human', 'rstar_ceiling_mouse', 'index', 'reliability_human_ref'])]
display(show.assign(text=[f'{v:.2f} ({lo:.2f}, {hi:.2f})' for v, lo, hi in zip(show.value, show.ci_low, show.ci_high)])
        .pivot_table(index=['contrast', 'subset'], columns='statistic', values='text', aggfunc='first')
        .reindex(pd.MultiIndex.from_product([list(index_genes), SUBSET_NAMES])))

## 6 · Part B: sensitivity of the conservation index (referee M2)

All genes of notebook 43's index set. Settings: winsorisation q × reliability convention
(notebook 43 computes reliability on unwinsorised values; "matched" uses the same winsorised
values as r). Intervals: gene bootstrap (seed 43, as published), module-block bootstrap (k = 100,
50, 200), external-donor bootstrap (Lake healthy and Census male snRNA donors resampled; our
specimens fixed) and donor + gene.

In [ ]:
SETTINGS = [(q, match) for q in Q_GRID for match in (False, True)]
B_STATS = ('index', 'index_external', 'rstar_cross', 'rstar_external', 'rstar_ceiling_human', 'rstar_ceiling_mouse')


def evaluate_settings(data, idx):
    out = np.empty((len(SETTINGS), len(B_STATS)))
    for i, (q, match) in enumerate(SETTINGS):
        result = subset_statistics(data, idx, q=q, match_reliability=match)
        out[i] = [result[s] for s in B_STATS]
    return out


def sensitivity_stage(contrast):
    genes = index_genes[contrast]
    n = len(genes)
    data = full_data(contrast, genes)
    lake_pd, sn_pd = reference_per_donor(contrast, genes)
    out = {'point': evaluate_settings(data, np.arange(n)),
           'gene': np.stack([evaluate_settings(data, idx) for idx in gene_draws(n, N_BOOT, SEED43)])}
    for k in (N_MODULES, *N_MODULES_SENSITIVITY):
        blocks = module_of[f'k{k}'].reindex(genes).to_numpy()
        out[f'module k={k}'] = np.stack([evaluate_settings(data, idx) for idx in block_draws(blocks, N_BOOT, SEED)])
    for name in ('donor', 'donor + gene'):
        rng = np.random.default_rng(SEED)
        draws = []
        for _ in range(N_BOOT):
            resampled = {**data, 'human_ref': resampled_gradient(lake_pd.to_numpy(), rng), 'mouse_ref': resampled_gradient(sn_pd.to_numpy(), rng)}
            idx = rng.integers(0, n, n) if name == 'donor + gene' else np.arange(n)
            draws.append(evaluate_settings(resampled, idx))
        out[name] = np.stack(draws)
    loo = []
    for donor in healthy_lake:       # leave one Lake donor out (S2 − S1 human-ceiling dependence)
        mean, noise = paired_gradient(lake_pd.drop(index=donor))
        loo.append(evaluate_settings({**data, 'human_ref': (mean.to_numpy(float), noise.to_numpy(float))}, np.arange(n)))
    out['leave one Lake donor out'] = np.stack(loo)
    return out


sensitivity = {contrast: cached_payload(f'sensitivity_{contrast}', lambda c=contrast: sensitivity_stage(c), root=cache,
                                        params={'settings': SETTINGS, 'stats': B_STATS, 'n_boot': N_BOOT, 'seeds': [SEED43, SEED],
                                                'k': [N_MODULES, *N_MODULES_SENSITIVITY]},
                                        inputs={'genes': list(index_genes[contrast]), 'ours': digest(ours), 'lake': digest(lake_counts),
                                                'sn': digest(sn_counts.sort_index(axis=1)), 'modules': digest(modules)}, code=code)
               for contrast in index_genes}
BOOTSTRAPS = ['gene', f'module k={N_MODULES}', *[f'module k={k}' for k in N_MODULES_SENSITIVITY], 'donor', 'donor + gene']
sens_rows = []
for contrast, stage in sensitivity.items():
    for boot in BOOTSTRAPS:
        low, high, missing = percentile_interval(stage[boot], max_missing=MAX_MISSING)
        for i, (q, match) in enumerate(SETTINGS):
            for version, stat in (('primary (ours)', 'index'), ('external (Lake vs snRNA male)', 'index_external')):
                j = B_STATS.index(stat)
                sens_rows.append({'contrast': contrast, 'version': version, 'winsor_q': q,
                                  'reliability': 'matched (winsorised)' if match else 'notebook 43 (unwinsorised)', 'bootstrap': boot,
                                  'index': stage['point'][i, j], 'ci_low': low[i, j], 'ci_high': high[i, j],
                                  'upper < 0.5': bool(high[i, j] < INDEX_STRONG),
                                  'ceiling_human': stage['point'][i, B_STATS.index('rstar_ceiling_human')],
                                  'ceiling_mouse': stage['point'][i, B_STATS.index('rstar_ceiling_mouse')]})
index_sensitivity = pd.DataFrame(sens_rows)
index_sensitivity.to_csv(tables / 'index_sensitivity.csv', index=False)

# Published setting check: q = 0.01, notebook 43 reliability, gene bootstrap (seed 43).
check = index_sensitivity[index_sensitivity.winsor_q.eq(Q_PUBLISHED) & index_sensitivity.reliability.str.startswith('notebook')
                          & index_sensitivity.bootstrap.eq('gene')]
for row in check.itertuples(index=False):
    target = published[published.contrast.eq(row.contrast) & published.version.str.startswith(
        'primary' if row.version.startswith('primary') else 'external: ')].iloc[0]
    if not all(np.isclose(a, b, rtol=0, atol=1e-9) for a, b in ((row.index, target['index']), (row.ci_low, target.ci_low),
                                                                 (row.ci_high, target.ci_high))):
        raise ValueError(f'Part B does not reproduce the published row: {row.contrast} {row.version}')
print('Part B reproduces the published rows at q = 0.01, notebook 43 reliability, gene bootstrap.')

summary_b = index_sensitivity.assign(text=[f'{v:.2f} ({lo:.2f}–{hi:.2f})' for v, lo, hi in
                                           zip(index_sensitivity['index'], index_sensitivity.ci_low, index_sensitivity.ci_high)])
display(summary_b[summary_b.bootstrap.isin(['gene', f'module k={N_MODULES}', 'donor', 'donor + gene'])]
        .pivot_table(index=['version', 'contrast', 'winsor_q', 'reliability'], columns='bootstrap', values='text', aggfunc='first')
        [['gene', f'module k={N_MODULES}', 'donor', 'donor + gene']])

primary_b = index_sensitivity[index_sensitivity.version.str.startswith('primary') & index_sensitivity.contrast.isin(['S2-S1', 'S3-early'])]
WORDING_STANDS = bool(primary_b['upper < 0.5'].all())
failures = index_sensitivity[~index_sensitivity['upper < 0.5']]
print(f'"Largely different genes" (every primary S2 − S1 / S3 − early upper bound < 0.5): {WORDING_STANDS}')
print(f'Upper bound range, primary index: {primary_b.ci_high.min():.2f}–{primary_b.ci_high.max():.2f}; '
      f'all versions and contrasts: max {index_sensitivity.ci_high.max():.2f}')
display(failures if len(failures) else pd.DataFrame({'note': ['no upper bound reaches 0.5']}))

### The S2 − S1 human ceiling rests on one dataset pair

Census human has no S2, so the S2 − S1 human ceiling is ours against Lake only. We report Lake S2
nuclei per donor, the leave-one-donor-out range, the donor-bootstrap interval and the human ceiling
at which the primary index's upper bound would reach 0.5 (the index scales as ceiling^−½; the
interval is assumed to scale with it).

In [ ]:
published_setting = SETTINGS.index((Q_PUBLISHED, False))
ceiling_rows = []
for contrast, stage in sensitivity.items():
    j_ceiling, j_index = B_STATS.index('rstar_ceiling_human'), B_STATS.index('index')
    loo = stage['leave one Lake donor out'][:, published_setting, :]
    d_low, d_high, _ = percentile_interval(stage['donor'][:, published_setting, j_ceiling][:, None])
    point = stage['point'][published_setting]
    _, i_high, _ = percentile_interval(stage['gene'][:, published_setting, j_index][:, None])
    ceiling_rows.append({'contrast': contrast, 'human ceiling': point[j_ceiling],
                         'leave-one-donor-out min': loo[:, j_ceiling].min(), 'leave-one-donor-out max': loo[:, j_ceiling].max(),
                         'donor bootstrap low': d_low[0], 'donor bootstrap high': d_high[0],
                         'index, leave-one-donor-out range': f'{loo[:, j_index].min():.3f}–{loo[:, j_index].max():.3f}',
                         'index upper bound': i_high[0],
                         'human ceiling at which the upper bound reaches 0.5': point[j_ceiling] * (i_high[0] / INDEX_STRONG) ** 2,
                         'human datasets giving a ceiling': 1 if contrast == 'S2-S1' else 2})
human_ceiling = pd.DataFrame(ceiling_rows)
human_ceiling.to_csv(tables / 'human_ceiling_dependence.csv', index=False)
lake_s2_nuclei = nuclei_table.loc[healthy_lake, ['S1', 'S2', 'S3']]
lake_s2_nuclei.to_csv(tables / 'lake_healthy_nuclei_per_segment.csv')
display(human_ceiling.round(3))
display(lake_s2_nuclei)

## 7 · Figures

Main panel set: noise-corrected correlations (cross-species and both ceilings), the conservation
index and direction agreement by strength bin, primary scheme, gene-bootstrap 95% bands.
"n.e." marks a bin where the statistic is not estimable (a reliability < 0.2). Direction agreement is
shown only for bins with at least 10 genes zonated in both species.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
# Okabe-Ito hues as in notebooks 41 and 43 (palette validated for colour-vision deficiency and contrast);
# dashed series carry a second encoding besides colour.
HUMAN, MOUSE, CROSS, INK, MUTED, ACCENT = '#D55E00', '#0072B2', '#009E73', '#333333', '#9A9A9A', '#AA4499'
TITLES = {'S2-S1': 'S2 − S1', 'S3-early': 'S3 − early', 'S3c-early': 'S3c − early (region-matched)'}
X = np.arange(len(BINS))


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=220)


def curve(ax, contrast, stat, color, label, *, scheme='max rank (primary)', dashed=False, offset=0., band=True):
    block = curves[curves.contrast.eq(contrast) & curves.scheme.eq(scheme) & curves.q.eq(Q_PUBLISHED)
                   & curves.bootstrap.eq('gene') & curves.statistic.eq(stat)].set_index('subset').reindex(BINS)
    if band:
        ax.fill_between(X + offset, block.ci_low.astype(float), block.ci_high.astype(float), color=color, alpha=.14, lw=0)
    ax.plot(X + offset, block.value.astype(float), color=color, lw=2, ls='--' if dashed else '-', marker='o', ms=4, label=label)
    return block.value.astype(float).to_numpy()


fig, axes = plt.subplots(3, 3, figsize=(13, 10.5), sharex=True, gridspec_kw={'height_ratios': [1.15, 1, .8]})
for col, contrast in enumerate(TITLES):
    ax = axes[0, col]
    ceiling_h = curve(ax, contrast, 'rstar_ceiling_human', HUMAN, 'within human: ours vs Lake')
    curve(ax, contrast, 'rstar_ceiling_mouse', MOUSE, 'within mouse: ours vs snRNA')
    curve(ax, contrast, 'rstar_cross', CROSS, 'human vs mouse: ours')
    curve(ax, contrast, 'rstar_external', CROSS, 'human vs mouse: Lake vs snRNA', dashed=True, band=False)
    for i in np.flatnonzero(np.isnan(ceiling_h)):
        ax.text(X[i], -.27, 'human\nceiling n.e.', color=HUMAN, ha='center', va='bottom', fontsize=6.5)
    ax.axhline(0, color=MUTED, lw=.6)
    ax.set_ylim(-.3, 1.25)
    ax.set_title(f'{"abc"[col]} · {TITLES[contrast]}', loc='left')
    ax = axes[1, col]
    index_values = curve(ax, contrast, 'index', INK, 'conservation index: ours')
    curve(ax, contrast, 'index_external', ACCENT, 'conservation index: external atlases', dashed=True, band=False)
    for i in np.flatnonzero(np.isnan(index_values)):
        ax.text(X[i], .02, 'n.e.', color=MUTED, ha='center', va='bottom', fontsize=7)
    overall = curves[curves.contrast.eq(contrast) & curves.scheme.eq('max rank (primary)') & curves.q.eq(Q_PUBLISHED)
                     & curves.bootstrap.eq('gene') & curves.statistic.eq('index') & curves.subset.eq('all genes')].value.iloc[0]
    ax.axhline(INDEX_STRONG, color=MUTED, lw=.8, ls=':')
    ax.axhline(overall, color=INK, lw=.7, ls='-.', alpha=.5)
    ax.text(X[0] - .3, overall + .02, f'all genes {overall:.2f}', color=INK, fontsize=6.5, alpha=.8)
    ax.axhline(0, color=MUTED, lw=.6)
    ax.set_ylim(-.25, 1.05)
    ax.set_title(f'{"def"[col]} · conservation index', loc='left')
    ax = axes[2, col]
    block = direction_agreement[direction_agreement.contrast.eq(contrast)].set_index('bin').reindex(BINS)
    block.loc[block['zonated in both'] < 10, ['share conserved', 'share low', 'share high']] = np.nan   # too few genes to show
    ax.errorbar(X, block['share conserved'], yerr=[block['share conserved'] - block['share low'], block['share high'] - block['share conserved']],
                fmt='o-', color=INK, lw=1.6, ms=4, capsize=0, label='zonated in both: same direction')
    ax.plot(X, block['raw sign agreement (all genes)'], color=MUTED, ls='--', marker='o', ms=3, lw=1.4, label='all genes: same sign')
    for i, n in enumerate(block['zonated in both']):
        ax.text(X[i], .27, f'n={int(n)}', ha='center', fontsize=6.5, color=MUTED)
    ax.axhline(.5, color=MUTED, lw=.6, ls=':')
    ax.set_ylim(.25, 1.02)
    ax.set_title(f'{"ghi"[col]} · direction agreement', loc='left')
    ax.set_xticks(X, BINS, rotation=30)
    ax.set_xlabel('Gradient-strength bin (percentile, cross-fitted)')
axes[0, 0].set_ylabel('Noise-corrected correlation r*')
axes[1, 0].set_ylabel('Index = r*(h, m) ÷ √(ceiling_h · ceiling_m)')
axes[2, 0].set_ylabel('Share in the same direction')
axes[0, 0].legend(frameon=False, fontsize=7, loc='lower right')
for row in (1, 2):
    axes[row, 0].legend(frameon=False, fontsize=7, loc='upper left')
fig.tight_layout()
save(fig, 'fig_conservation_by_strength')
plt.show()
plt.close(fig)

# Supplementary: binning schemes (selection effect and asymmetry).
SCHEME_COLORS = {'max rank (primary)': INK, 'mean rank': CROSS, 'human rank only': HUMAN, 'mouse rank only': MOUSE,
                 'same-data |our mouse| (referee)': ACCENT}
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
for col, contrast in enumerate(TITLES):
    for k, (scheme, color) in enumerate(SCHEME_COLORS.items()):
        for row, stat in enumerate(('r_cross', 'index')):
            curve(axes[row, col], contrast, stat, color, scheme, scheme=scheme, dashed=scheme.startswith('same-data'),
                  offset=(k - 2) * .07, band=False)
    axes[0, col].set_title(f'{"abc"[col]} · {TITLES[contrast]}: raw r, our human vs our mouse', loc='left')
    axes[1, col].set_title(f'{"def"[col]} · conservation index', loc='left')
    axes[1, col].set_xticks(X, BINS, rotation=30)
    axes[1, col].set_xlabel('Strength bin (percentile)')
    for row in (0, 1):
        axes[row, col].axhline(0, color=MUTED, lw=.6)
axes[0, 0].set_ylabel('Raw Pearson r')
axes[1, 0].set_ylabel('Conservation index')
axes[0, 0].legend(frameon=False, fontsize=7, loc='upper left')
fig.tight_layout()
save(fig, 'fig_strength_schemes')
plt.show()
plt.close(fig)

# Supplementary: index sensitivity (point and 95% interval per setting and bootstrap).
BOOT_COLORS = {'gene': INK, f'module k={N_MODULES}': CROSS, 'donor + gene': ACCENT}
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
for ax, version in zip(axes, ('primary (ours)', 'external (Lake vs snRNA male)')):
    block = index_sensitivity[index_sensitivity.version.eq(version) & index_sensitivity.bootstrap.isin(BOOT_COLORS)
                              & index_sensitivity.reliability.str.startswith('matched')].reset_index(drop=True)
    block = (block.assign(order_c=block.contrast.map({k: i for i, k in enumerate(TITLES)}),
                          order_b=block.bootstrap.map({k: i for i, k in enumerate(BOOT_COLORS)}))
             .sort_values(['order_c', 'winsor_q', 'order_b']).reset_index(drop=True))
    y = np.arange(len(block))[::-1]
    for boot, color in BOOT_COLORS.items():
        m = block.bootstrap.eq(boot).to_numpy()
        ax.hlines(y[m], block.ci_low[m], block.ci_high[m], color=color, lw=1.8, label=boot)
        ax.scatter(block['index'][m], y[m], color=color, s=16, zorder=3)
    ax.axvline(INDEX_STRONG, color=MUTED, ls='--', lw=.9)
    ax.set_xlim(-.15, .7)
    ax.set_title(f'{"ab"[version.startswith("external")]} · {version}', loc='left')
    ax.set_xlabel('Conservation index (95% interval)')
    ax.set_yticks(y, [f'{TITLES[c].split(" (")[0]} · q = {q:g}' if b == 'gene' else '' for c, q, b in
                      zip(block.contrast, block.winsor_q, block.bootstrap)], fontsize=7)
axes[0].legend(frameon=False, fontsize=7, loc='lower right')
fig.tight_layout()
save(fig, 'fig_index_sensitivity')
plt.show()
plt.close(fig)

## 8 · Decisions and run record

In [ ]:
headline = {}
for contrast in index_genes:
    block = curves[curves.contrast.eq(contrast) & curves.scheme.eq('max rank (primary)') & curves.q.eq(Q_PUBLISHED) & curves.bootstrap.eq('gene')]
    for subset in ('BOTTOM (<80%)', 'TOP (≥95%)', BINS[-1], 'all genes'):
        for stat in ('rstar_cross', 'rstar_ceiling_human', 'rstar_ceiling_mouse', 'index', 'index_external'):
            v = block[block.subset.eq(subset) & block.statistic.eq(stat)].iloc[0]
            headline[f'{contrast} · {subset} · {stat}'] = f'{v.value:.3f} ({v.ci_low:.3f}, {v.ci_high:.3f})'
decisions = pd.Series({'notebook 43 reproduced': True, 'rule (i) concentration (S2−S1 and S3−early, gene and module)': RULE_I,
                       'rule (ii) BOTTOM reproducible but not shared': RULE_II, 'BOTTOM not reliably zonated (any primary contrast)': NOT_ZONATED,
                       'title verdict': TITLE_VERDICT, '"largely different genes" stands under every sensitivity': WORDING_STANDS,
                       'max upper bound, primary index (S2−S1, S3−early)': float(primary_b.ci_high.max()),
                       'genes in "reproducible, not shared" bins': gene_counts.loc['reproducible, not shared'].to_dict()
                       if 'reproducible, not shared' in gene_counts.index else {},
                       'genes in "not reliably zonated" bins': gene_counts.loc['not reliably zonated'].to_dict()
                       if 'not reliably zonated' in gene_counts.index else {},
                       'settings with an upper bound ≥ 0.5 (primary index)': '; '.join(
                           f'{r.contrast} q={r.winsor_q:g} {r.reliability.split(" ")[0]} {r.bootstrap} ({r.ci_high:.2f})'
                           for r in index_sensitivity[index_sensitivity.version.str.startswith('primary') & ~index_sensitivity['upper < 0.5']].itertuples()),
                       'post hoc (mouse-strength bins, full Lake): BOTTOM human ceiling / index, S2−S1': '{:.2f} / {:.2f}'.format(
                           *posthoc['S2-S1|point'][SUBSET_NAMES.index('BOTTOM (<80%)'), [STATS.index('rstar_ceiling_human'), STATS.index('index')]])})
decisions.to_csv(tables / 'decisions.csv')
display(decisions.to_frame('value'))
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'labels': 'reviewed (notebook 13)',
            'lake_folds': lake_fold.to_dict(), 'snRNA_male_folds': sn_fold.to_dict(),
            'decisions': decisions.astype(str).to_dict(), 'headline': headline,
            'human_ceiling': human_ceiling.round(4).to_dict(orient='records')}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest['decisions'], indent=2))

### Reading the results (written after the run; numbers from the tables above)

- **Guard.** Notebook 43's published index is reproduced exactly (8 of 8 rows), by its own function
  and by this notebook's estimator.
- **Agreement rises with gradient strength.** With strength cross-fitted on external donors, the
  conservation index is about 0 below the 80th percentile (S3 − early 0.03, 95% CI −0.03 to 0.08;
  S2 − S1 cross r* −0.05) and 0.43–0.49 in the top 5% (top 1%: 0.55–0.58, CI from about 0.34 to
  0.74). Rule (i) holds in every contrast under gene and module-block bootstraps. No bin reaches a
  lower bound of 0.5, so even the strongest gradients are only partly shared.
- **Weaker gradients.** For S3 − early and S3c − early, the bottom 80% replicate within each species
  (ceilings about 0.6 human and 0.78–0.80 mouse) but not across species: rule (ii) holds. For
  S2 − S1, the human ceiling of the bottom 80% is not estimable: a Lake fold of 3–4 donors has
  reliability < 0.2 there. By the pre-registered rule this makes the S2 − S1 weak genes "not
  reliably zonated", and the verdict is the reworded claim. The post-hoc diagnostic with the full
  Lake reference (bins on mouse strength) gives an S2 − S1 bottom-80% human ceiling of 0.68 with an
  index of −0.11, so the missing ceiling reflects the halved reference rather than absent human
  zonation; it does not change the pre-registered verdict.
- **Asymmetry.** Binned on human strength, the weaker genes keep some agreement (S3 − early index
  0.18); binned on mouse strength they keep none (0.00). Genes weakly zonated in human still carry
  a faint mouse-like gradient; genes weakly zonated in mouse carry no human-like gradient.
- **Direction.** Among genes zonated in both species, the share with the same direction rises from
  about 0.5–0.6 in the 50–80% bin to 0.84–0.89 in the top 1%.
- **Index sensitivity.** Matching the reliability to the winsorised values changes the primary index
  by ≤ 0.01 and the external index by ≤ 0.03. Winsorisation moves it: q = 0, 0.01, 0.05 give 0.25, 0.17, 0.09 (S2 − S1) and 0.35, 0.28,
  0.19 (S3 − early); the external index stays 0.25–0.31. Gene, donor and donor + gene intervals stay
  below 0.5 (largest 0.45). Module-block intervals are four to five times wider, and for S3 − early and
  S3c − early (q ≤ 0.01) their upper bounds reach 0.50–0.54, so the pre-registered "every upper bound
  < 0.5" rule fails there. Donor-bootstrap percentile intervals are shifted relative to the point
  estimate: resampling 7 donors with replacement duplicates donors, which understates within-reference
  noise and so overstates reliability; the leave-one-donor-out range is the cleaner donor check.
- **S2 − S1 human ceiling.** One dataset pair (ours vs Lake; Lake S2 nuclei 59–780 per donor). The
  leave-one-donor-out range is 0.63–0.68 (index 0.166–0.173); the ceiling would have to fall to 0.13
  for the index's upper bound to reach 0.5.